# Reality Check #1 — Bonus notebook

## Why your computer has never seen the number 0.1

*Lecture 02 · Programming 1 · Fundamentals and Methods of Computer Science*

This notebook is **optional**. It is not part of the exercise ZIP and nothing in it
is examined as a story. What **is** examined is the behaviour it demonstrates:
how Python stores `float` values, how `round()` actually rounds, and why you
compare floats with a tolerance instead of `==`.

Run the cells top to bottom. Everything here uses only the standard library.

---
## 1. The question we asked in the lecture

In school mathematics, $0.1 + 0.2 = 0.3$. Ask Python.

In [ ]:
print(0.1 + 0.2 == 0.3)

In [ ]:
print(0.1 + 0.2)

That trailing `4` is not a bug, and it is not Python being sloppy. It is the
honest answer to a question we did not realise we were asking.

---
## 2. What Python actually stores

A `float` is a **binary** fraction with a fixed number of bits (64, on every machine
you will use). Ask for the exact decimal value of the thing Python stored when you
typed `0.1`:

In [ ]:
from decimal import Decimal

print(Decimal(0.1))

Your computer has never seen the number 0.1. It stored the closest 64-bit binary
fraction it has, and that number is slightly bigger than 0.1.

The reason is the same one that makes $1/3 = 0.333\ldots$ impossible to write down
exactly in decimal: a fraction only terminates if its denominator is built from the
prime factors of the base. In base 2 that means powers of two — a half, a quarter, an
eighth are exact, a tenth is not. So one tenth repeats forever in binary, and a float
has a fixed number of bits, so it gets cut off.

That is the whole mechanism. Everything below follows from it.

*(If you ever want to look at the bits themselves, `(0.1).hex()` prints the stored
value exactly. You will not need it here.)*

---
## 3. So how do you compare floats?

Not with `==`. Use a tolerance. The standard library has one ready:

In [ ]:
import math

a = 0.1 + 0.2
b = 0.3

print("a == b           :", a == b)
print("math.isclose(a,b):", math.isclose(a, b))
print("difference       :", abs(a - b))

---
## 4. `round()` is not what you think either

Python's `round()` uses **round-half-to-even** (often called *banker's rounding*):
a value exactly halfway between two options goes to the **even** one. Over many
values this removes the systematic upward bias that "always round half up" has.

In [ ]:
for x in [0.5, 1.5, 2.5, 3.5, 4.5]:
    print(f"round({x}) = {round(x)}")

Note that `round(2.5)` is `2`, not `3`. This surprises almost everyone once.

And rounding is **not** the same as cutting digits off. Cutting off is called
**truncation**:

In [ ]:
x = 1.9999

print("round(x, 2)    :", round(x, 2))       # nearest
print("truncated to 2 :", int(x * 100) / 100)  # lopped off

Rounding is unbiased: errors go up about as often as they go down, and they cancel.
Truncation is biased: the error is **always** in the same direction. It never cancels.
It accumulates.

That difference cost a stock exchange half its index.

---
## 5. The Vancouver Stock Exchange, 1982–1983

In January 1982 the Vancouver Stock Exchange launched a new index at **1,000.000**.
It was computed to four decimal places, displayed to three — and the program
**truncated** the fourth digit instead of rounding it. The index was recalculated
after every trade, roughly 2,800–3,000 times a day.

Each truncation loses, on average, half of a thousandth. Almost nothing.
Multiply it by 3,000 a day, for 22 months.

Let us simulate it. Same random market, two programs: one rounds, one truncates.

In [ ]:
import random

def truncate(value, digits=3):
    # Cut the value off after `digits` decimals - no rounding.
    factor = 10 ** digits
    return int(value * factor) / factor


def simulate(months=22, recalcs_per_day=2800, trading_days_per_month=21, seed=42):
    # Run the same sequence of market moves through both programs.
    random.seed(seed)
    rounded, truncated = 1000.0, 1000.0
    history = []

    for month in range(1, months + 1):
        for _ in range(recalcs_per_day * trading_days_per_month):
            # One trade nudges the index by a tiny amount, up or down.
            move = random.uniform(-0.01, 0.01)
            rounded = round(rounded + move, 3)
            truncated = truncate(truncated + move, 3)
        history.append((month, rounded, truncated))

    return history


history = simulate()

print(f"{'month':>6} {'rounded':>12} {'truncated':>12} {'gap':>10}")
for month, r, t in history:
    print(f"{month:>6} {r:>12.3f} {t:>12.3f} {r - t:>10.3f}")

The `rounded` column wanders, as a market should. The `truncated` column slides
downwards and never recovers — not because the market fell, but because a program
threw away a fraction of a thousandth, several thousand times a day, always in the
same direction.

The real numbers: in November 1983, after 22 months, the index read **524.811**.
Recalculated correctly, it was **1,098.892**. Nobody had stolen anything. Nobody had
made a bad trade. A programmer had chosen truncation over rounding.

> **A note on the model.** Our simulation drifts by about 29 points a month; the
> contemporary reports put the real drift nearer 20. The gap is the model's, not the
> story's: we assume every one of the 2,800 daily recalculations feeds its truncated
> result into the next, which is a simplification of how the exchange actually
> chained its updates. Getting the mechanism right and the magnitude roughly right is
> what a simulation is for — and noticing that your model is 50% off is a skill worth
> more than the model.

### Optional: see it

Needs `matplotlib`, which ships with Anaconda.

In [ ]:
import matplotlib.pyplot as plt

months = [h[0] for h in history]
rounded_vals = [h[1] for h in history]
truncated_vals = [h[2] for h in history]

plt.figure(figsize=(8, 4.5))
plt.plot(months, rounded_vals, label="rounded", linewidth=2)
plt.plot(months, truncated_vals, label="truncated", linewidth=2, linestyle="--")
plt.xlabel("month")
plt.ylabel("index value")
plt.title("Same market, two programs")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## 6. What you actually do about it

Three rules. They are worth more to you than the story.

**1. Never compare floats with `==`.** Use `math.isclose()`, or compare the
difference against a tolerance you chose deliberately.

**2. Never use `float` for money.** Use `decimal.Decimal` (or count in whole cents
as integers). `Decimal` does arithmetic the way your accountant does — in base 10,
exactly.

In [ ]:
from decimal import Decimal

print("float  :", 0.1 + 0.2 == 0.3)
print("Decimal:", Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))

In [ ]:
# Careful: Decimal(0.1) inherits the float's error. Always pass a STRING.
print('Decimal(0.1)   =', Decimal(0.1))
print('Decimal("0.1") =', Decimal("0.1"))

**3. Rounding and truncating are different operations.** Pick one on purpose, and
know which one your language does by default. `round()` rounds half to even;
`int()` truncates towards zero; `math.floor()` goes down; `math.ceil()` goes up.

In [ ]:
import math

for x in [2.7, -2.7]:
    print(f"x = {x:>5}  round={round(x):>3}  int={int(x):>3}  "
          f"floor={math.floor(x):>3}  ceil={math.ceil(x):>3}")

---
## Sources

- Michael Eisen, *Amazon's \$23,698,655.93 book about flies* — next week's story.
- Toronto Star, 29 November 1983, on the corrected Vancouver index (1,098.892).
- Wall Street Journal, 8 November 1983, on the index launch and the recalculation.
- Python documentation, *Floating Point Arithmetic: Issues and Limitations*:
  https://docs.python.org/3/tutorial/floatingpoint.html
- Python documentation, `decimal` module: https://docs.python.org/3/library/decimal.html